# Module 32 — What an MCP server exposes, and who controls each thing

**THE ONE IDEA:** MCP has three capabilities, and the distinction that actually matters in
a design review is **control dynamics** — *who decides this gets used?*

| capability | what it is | **controlled by** |
|---|---|---|
| **Tool** | a function with side effects | the **model** — it decides to call it |
| **Resource** | read-only data at a URI | the **application** — the host decides what to attach |
| **Prompt** | a reusable template | the **user** — picked from a menu, e.g. a slash command |

That column is the answer to "what is the difference between a tool and a resource?" It
is not *read vs write*. It is **who is in charge**.

> **Version note.** This block is built on **JSON-RPC 2.0** and the tools/resources/prompts
> core, which are stable. Your ladder README says a 2026-07-28 revision made MCP stateless
> and added MRTR. **I could not verify that against a primary source**, so nothing here
> depends on it. Check the spec before relying on the details in module 34.

No `mcp` package and no API key — the protocol is JSON, so we implement it.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from _tools import _POLICY, run_tool

class MCPServer:
    """A minimal MCP server. The protocol is JSON-RPC 2.0 over a byte stream —
    that is genuinely all it is, so it fits in one class."""
    NAME, VERSION = "bank-policy-server", "0.1.0"

    def __init__(self):
        self.tools = {
            "search_policy": {
                "description": "Search bank policy documents by keyword.",
                "inputSchema": {"type": "object",
                                "properties": {"query": {"type": "string"}},
                                "required": ["query"]}},
            "calculate": {
                "description": "Evaluate an arithmetic expression.",
                "inputSchema": {"type": "object",
                                "properties": {"expression": {"type": "string"}},
                                "required": ["expression"]}}}
        self.resources = {f"policy://{k}": {"name": k, "mimeType": "text/plain", "text": v}
                          for k, v in _POLICY.items()}
        self.prompts = {"underwrite": {
            "description": "Underwriting review template",
            "arguments": [{"name": "case_id", "required": True}]}}

## The three listings

Note the shapes. A tool advertises an **inputSchema** because the model must construct
arguments. A resource advertises a **URI and mimeType** because the host must decide
whether to attach it. A prompt advertises **arguments** because a person fills them in.

In [ ]:
server = MCPServer()

def rpc(method, params=None, _id=1):
    """Every MCP message is this shape. Nothing more exotic than JSON-RPC 2.0."""
    return {"jsonrpc": "2.0", "id": _id, "method": method, "params": params or {}}

print("REQUEST :", json.dumps(rpc("tools/list")))
print("\ntools/list ->")
for n, t in server.tools.items():
    print(f"   {n:15} {t['description'][:44]:44} args={list(t['inputSchema']['properties'])}")

print("\nresources/list ->")
for uri, r in list(server.resources.items())[:4]:
    print(f"   {uri:22} {r['mimeType']:12} {r['text'][:34]}")

print("\nprompts/list ->")
for n, p in server.prompts.items():
    print(f"   {n:15} args={[a['name'] for a in p['arguments']]}")

## Calling each one

In [ ]:
def handle(req):
    m, p = req["method"], req.get("params", {})
    if m == "tools/call":
        out = run_tool(p["name"], p["arguments"])
        return {"jsonrpc": "2.0", "id": req["id"],
                "result": {"content": [{"type": "text", "text": out}],
                           "isError": out.startswith("ERROR")}}
    if m == "resources/read":
        r = server.resources.get(p["uri"])
        if not r:                                   # JSON-RPC error, not an exception
            return {"jsonrpc": "2.0", "id": req["id"],
                    "error": {"code": -32602, "message": f"unknown resource {p['uri']}"}}
        return {"jsonrpc": "2.0", "id": req["id"],
                "result": {"contents": [{"uri": p["uri"], "text": r["text"]}]}}
    if m == "prompts/get":
        return {"jsonrpc": "2.0", "id": req["id"], "result": {"messages": [
            {"role": "user", "content": {"type": "text", "text":
             f"Review case {p['arguments']['case_id']} against LTV, income and deposit policy."}}]}}
    return {"jsonrpc": "2.0", "id": req["id"],
            "error": {"code": -32601, "message": f"method not found: {m}"}}

for req in [rpc("tools/call", {"name": "search_policy", "arguments": {"query": "erc"}}),
            rpc("resources/read", {"uri": "policy://ltv"}, 2),
            rpc("prompts/get", {"name": "underwrite", "arguments": {"case_id": "C-1002"}}, 3),
            rpc("resources/read", {"uri": "policy://nope"}, 4)]:
    res = handle(req)
    key = "error" if "error" in res else "result"
    print(f"  {req['method']:16} -> {key}: {json.dumps(res[key])[:88]}")

## The control-dynamics question

In [ ]:
print(f"{'capability':11} {'who decides it is used':26} {'consequence'}")
print("-" * 84)
for cap, who, why in [
 ("Tool",     "the MODEL, at runtime",     "needs a schema; needs a human gate if it writes"),
 ("Resource", "the APPLICATION / host",    "read-only by design; host chooses what to attach"),
 ("Prompt",   "the USER, from a menu",     "a slash command; the server ships the workflow")]:
    print(f"{cap:11} {who:26} {why}")

print("""
LESSON - 'what is the difference between a tool and a resource?' is a control
question, not a read/write one. A read_file TOOL and a file:// RESOURCE can
return identical bytes. The difference is that the MODEL decides to call the
tool, while the HOST decides to attach the resource.

Why that matters in practice:

  - resources are SAFER by construction. No side effects, and the model cannot
    reach for one on its own - so a poisoned prompt cannot make it pull a file
    the host did not offer.
  - tools carry the risk, which is why modules 14 and 16 apply to them and not
    to resources: human gates on writes, capability isolation on reads of
    attacker-controlled data.
  - prompts are how a server author ships a WORKFLOW rather than a capability.
    The user picks it; neither the model nor the host chooses for them.

Everything above is plain JSON-RPC 2.0 over a byte stream. There is no magic in
MCP - the value is that EVERYONE AGREED on these shapes, which is what turns
M hosts x N tools into M + N.""")

---

**Next:** `33_mcp_client_and_lifecycle.ipynb`